In [ ]:
# -*- coding: utf-8 -*-
"""
TCNN / AUTOENCODER RESIDUAL — REFERÊNCIA SEMPRE SAUDÁVEL

Objetivo:
- A referência base é SEMPRE a curva saudável em REF_TEMP.
- A rede deve aprender apenas a compensação térmica.
- A rede NÃO deve transformar dano 1 ou dano 2 em saudável.
- Para preservar dano, o alvo é:

    alvo = referência saudável em REF_TEMP
           + resíduo da curva original em relação à saudável da mesma temperatura

Exemplo:

    curva D2 em 55°C

    saudável em 55°C = H(55)
    saudável em REF_TEMP = H(REF)

    resíduo de dano aproximado:
        R_dano = curva_D2_55 - H(55)

    alvo da rede:
        curva_alvo = H(REF) + R_dano

Assim, a rede aprende:

    curva_D2_55  --->  H(REF) + assinatura_de_dano

e não:

    curva_D2_55  --->  curva saudável pura

"""

# ============================================================
# 1) IMPORTS
# ============================================================

import os
import re
import time
import copy
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

try:
    from IPython.display import display
except Exception:
    display = print

warnings.filterwarnings("ignore", category=UserWarning)


# ============================================================
# 2) PARÂMETROS
# ============================================================

ARQ_BASE = "base-completo--.pkl"

REF_TEMP = 30

FREQ_MIN_KHZ = 40
FREQ_MAX_KHZ = 50

OUTPUT_DIR = (
    f"TCNN_REF_SAUDAVEL_"
    f"{REF_TEMP}C_{FREQ_MIN_KHZ}-{FREQ_MAX_KHZ}kHz"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ---------------- PARK PARA COMPARAÇÃO ----------------

PARK_MAX_SHIFT_FRAC = 0.10
PARK_NSTEPS = 101
PARK_SMOOTH_WIN = 1


# ---------------- TCNN ----------------

EPOCHS = 350
BATCH_SIZE = 16
LR = 1e-3
PATIENCE = 60

LATENT_DIM = 64

# Quanto da correção prevista será aplicada.
# Se ainda estiver apagando dano, teste 0.80 ou 0.90.
ALPHA_COMP = 1.00

# Limite da correção residual no espaço normalizado.
RESIDUAL_PERCENTILE = 99.5
RESIDUAL_SCALE_MIN = 0.20
RESIDUAL_SCALE_MAX = 5.00

# Pesos da função de custo
LAMBDA_CURVE = 1.00

# Muito importante:
# força a rede a aprender o DELTA térmico,
# e não simplesmente reconstruir qualquer curva.
LAMBDA_DELTA_TARGET = 2.0
LAMBDA_DELTA_DERIV = 0.30

# Preserva a assinatura residual de dano.
LAMBDA_SIGNATURE = 0.80
LAMBDA_SIGNATURE_DERIV = 0.25

# Mantém forma global parecida com o alvo.
LAMBDA_CORR = 0.10

# Evita correções exageradas.
LAMBDA_DELTA_ENERGY = 0.003

# Mantém curva saudável em REF_TEMP quase igual à entrada.
LAMBDA_IDENTITY_REF = 0.80

# Auxiliares.
# A falha NÃO entra como input.
# A temperatura NÃO entra como input.
# Eles são apenas cabeças auxiliares para organizar o espaço latente.
LAMBDA_DAMAGE = 0.08
LAMBDA_TEMP = 0.04
USE_TEMP_AUX_TARGET = True

NUM_WORKERS = 0

HIST_BINS = 18

# Temperatura usada para as curvas exemplo.
TEMP_ESCOLHIDA = 55
DANOS_PLOTAR = [0, 1, 2]
OCORRENCIA_CURVA = 0

# Se quiser escolher manualmente as temperaturas das barras:
# TEMPERATURAS_BARRAS = [-10, 10, 20, 30, 40, 50, 60, 70]
TEMPERATURAS_BARRAS = None
N_TEMPS_BARRAS = 8

np.random.seed(42)
torch.manual_seed(42)


# ============================================================
# 3) FUNÇÕES GERAIS
# ============================================================

def extract_freq_hz(col):
    m = re.match(r"^f_(\d+(?:\.\d+)?)Hz$", str(col))
    return float(m.group(1)) if m else None


def get_freq_columns(df, fmin_khz, fmax_khz):
    cols = []
    freqs = []

    for c in df.columns:
        f = extract_freq_hz(c)

        if f is not None:
            f_khz = f / 1e3

            if fmin_khz <= f_khz <= fmax_khz:
                cols.append(c)
                freqs.append(f)

    order = np.argsort(freqs)

    fcols = [cols[i] for i in order]
    fhz = np.array(freqs, dtype=float)[order]

    return fcols, fhz


def aplicar_estilo_artigo():
    plt.rcParams.update({
        "font.family": "Times New Roman",
        "font.size": 18,
        "axes.labelsize": 20,
        "axes.titlesize": 20,
        "xtick.labelsize": 17,
        "ytick.labelsize": 17,
        "legend.fontsize": 14,
        "figure.dpi": 300,
        "savefig.dpi": 300,
        "pdf.fonttype": 42,
        "ps.fonttype": 42
    })


def salvar_figura(fig, nome_base, output_dir=OUTPUT_DIR, dpi=600):
    os.makedirs(output_dir, exist_ok=True)

    png_path = os.path.join(output_dir, f"{nome_base}.png")
    pdf_path = os.path.join(output_dir, f"{nome_base}.pdf")

    fig.savefig(
        png_path,
        dpi=dpi,
        bbox_inches="tight",
        facecolor="white"
    )

    fig.savefig(
        pdf_path,
        bbox_inches="tight",
        facecolor="white"
    )

    print(f"Figura salva em PNG: {png_path}")
    print(f"Figura salva em PDF: {pdf_path}")

    return png_path, pdf_path


def moving_average(arr, win):
    if win <= 1 or win % 2 == 0:
        return arr.copy()

    pad = win // 2
    arr_pad = np.pad(arr, (pad, pad), mode="edge")

    kernel = np.ones(win) / win
    smooth = np.convolve(arr_pad, kernel, mode="valid")

    return smooth[:len(arr)]


def formatar_temp(T):
    if float(T).is_integer():
        return f"{int(T)}"
    return f"{T:.1f}"


# ============================================================
# 4) REFERÊNCIA SAUDÁVEL
# ============================================================

def get_healthy_references_by_temperature(df, fcols, ref_temp):
    """
    Cria uma referência saudável para cada temperatura disponível.

    healthy_by_temp[T] = mediana das curvas com falha = 0 naquela temperatura.

    A referência principal é sempre:
        y_ref_healthy = saudável em REF_TEMP

    Se REF_TEMP exata não existir, usa a saudável mais próxima.
    """

    if "falha" not in df.columns:
        raise ValueError("O DataFrame precisa ter a coluna 'falha'.")

    if "temperatura_c" not in df.columns:
        raise ValueError("O DataFrame precisa ter a coluna 'temperatura_c'.")

    df_h = df[df["falha"] == 0].copy()

    if len(df_h) == 0:
        raise ValueError("Não há curvas saudáveis, isto é, falha = 0.")

    healthy_by_temp = {}

    temps_h = np.array(
        sorted(df_h["temperatura_c"].unique()),
        dtype=float
    )

    for T in temps_h:
        pool = df_h.loc[
            np.isclose(df_h["temperatura_c"], T),
            fcols
        ].to_numpy(float)

        healthy_by_temp[float(T)] = np.median(pool, axis=0)

    if np.any(np.isclose(temps_h, ref_temp)):
        ref_temp_used = float(temps_h[np.argmin(np.abs(temps_h - ref_temp))])
    else:
        ref_temp_used = float(temps_h[np.argmin(np.abs(temps_h - ref_temp))])
        print(
            f"AVISO: não existe saudável exatamente em {ref_temp} °C. "
            f"Usando {ref_temp_used} °C como referência saudável."
        )

    y_ref_healthy = healthy_by_temp[ref_temp_used]

    return healthy_by_temp, temps_h, y_ref_healthy, ref_temp_used


def get_nearest_healthy_curve(healthy_by_temp, healthy_temps, temperatura):
    """
    Retorna a curva saudável mais próxima da temperatura pedida.
    """

    healthy_temps = np.asarray(healthy_temps, dtype=float)

    temp_used = float(
        healthy_temps[np.argmin(np.abs(healthy_temps - temperatura))]
    )

    return healthy_by_temp[temp_used], temp_used


# ============================================================
# 5) ALVOS DA TCNN
# ============================================================

def construir_targets_ref_saudavel_preservando_dano(
    df,
    fcols,
    healthy_by_temp,
    healthy_temps,
    y_ref_healthy
):
    """
    Constrói os alvos da rede usando SEMPRE referência saudável.

    Para cada curva X_i em temperatura T_i:

        H_T = curva saudável na mesma temperatura T_i
        H_REF = curva saudável em REF_TEMP

        assinatura_i = X_i - H_T

        alvo_i = H_REF + assinatura_i

    Então:

        delta_térmico_i = alvo_i - X_i
                         = H_REF - H_T

    Repare que o delta alvo NÃO depende diretamente do dano.
    Isso é bom, porque o trabalho da rede vira aprender a correção térmica.
    """

    X = df[fcols].to_numpy(float)

    Y = np.zeros_like(X)
    H_same_T = np.zeros_like(X)

    temp_healthy_used = []

    for i, (_, row) in enumerate(df.iterrows()):
        T = float(row["temperatura_c"])

        h_T, T_used = get_nearest_healthy_curve(
            healthy_by_temp=healthy_by_temp,
            healthy_temps=healthy_temps,
            temperatura=T
        )

        assinatura_dano = X[i] - h_T

        Y[i] = y_ref_healthy + assinatura_dano
        H_same_T[i] = h_T

        temp_healthy_used.append(T_used)

    temp_healthy_used = np.asarray(temp_healthy_used, dtype=float)

    D_target = Y - X

    return X, Y, D_target, H_same_T, temp_healthy_used


# ============================================================
# 6) MÉTRICAS
# ============================================================

def rmsd(y, ref):
    y = np.asarray(y, dtype=float)
    ref = np.asarray(ref, dtype=float)

    return float(np.sqrt(np.mean((y - ref) ** 2)))


def ccdm(y, ref):
    y = np.asarray(y, dtype=float)
    ref = np.asarray(ref, dtype=float)

    y0 = y - np.mean(y)
    r0 = ref - np.mean(ref)

    num = float(np.sum(y0 * r0))
    den = float(np.sqrt(np.sum(y0 ** 2) * np.sum(r0 ** 2)) + 1e-18)

    corr = num / den

    return float(1 - corr)


def corr_loss_batch(y_pred, y_true):
    yp = y_pred.squeeze(1)
    yt = y_true.squeeze(1)

    yp0 = yp - yp.mean(dim=1, keepdim=True)
    yt0 = yt - yt.mean(dim=1, keepdim=True)

    num = torch.sum(yp0 * yt0, dim=1)

    den = torch.sqrt(
        torch.sum(yp0 ** 2, dim=1) *
        torch.sum(yt0 ** 2, dim=1) +
        1e-12
    )

    corr = num / den

    return torch.mean(1.0 - corr)


def derivative_loss(y_pred, y_true):
    dy_pred = y_pred[:, :, 1:] - y_pred[:, :, :-1]
    dy_true = y_true[:, :, 1:] - y_true[:, :, :-1]

    return F.smooth_l1_loss(dy_pred, dy_true)


def delta_energy_loss(delta):
    return torch.mean(delta ** 2)


def calcular_metricas_com_preservacao(
    df_curvas,
    df_original,
    fcols,
    y_ref_healthy,
    healthy_by_temp,
    healthy_temps,
    metodo
):
    """
    Métricas principais:

    RMSD e CCDM:
        curva compensada contra referência saudável em REF_TEMP.

    DamageResidual_RMSD e DamageResidual_CCDM:
        verifica se a assinatura de dano foi preservada.

    A assinatura esperada do dano é:

        original - saudável_na_mesma_temperatura

    A assinatura depois da compensação é:

        compensada - saudável_em_REF_TEMP

    Se a compensação for boa, essas duas assinaturas devem ser parecidas.
    """

    X_comp = df_curvas[fcols].to_numpy(float)
    X_orig = df_original[fcols].to_numpy(float)

    df_out = df_curvas.copy()

    rmsd_list = []
    ccdm_list = []

    damage_res_rmsd = []
    damage_res_ccdm = []

    alteracao_rmsd = []

    for i, (_, row) in enumerate(df_original.iterrows()):
        T = float(row["temperatura_c"])

        h_T, _ = get_nearest_healthy_curve(
            healthy_by_temp=healthy_by_temp,
            healthy_temps=healthy_temps,
            temperatura=T
        )

        y_comp = X_comp[i]
        y_orig = X_orig[i]

        rmsd_list.append(rmsd(y_comp, y_ref_healthy))
        ccdm_list.append(ccdm(y_comp, y_ref_healthy))

        assinatura_esperada = y_orig - h_T
        assinatura_saida = y_comp - y_ref_healthy

        damage_res_rmsd.append(
            rmsd(assinatura_saida, assinatura_esperada)
        )

        damage_res_ccdm.append(
            ccdm(assinatura_saida, assinatura_esperada)
        )

        alteracao_rmsd.append(
            rmsd(y_comp, y_orig)
        )

    df_out["RMSD"] = rmsd_list
    df_out["CCDM"] = ccdm_list

    df_out["DamageResidual_RMSD"] = damage_res_rmsd
    df_out["DamageResidual_CCDM"] = damage_res_ccdm

    df_out["Alteracao_RMSD"] = alteracao_rmsd

    df_out["Metodo"] = metodo

    return df_out


# ============================================================
# 7) PARK
# ============================================================

def shift_interp(x, fHz, tau):
    f_shift = fHz + tau

    return np.interp(
        fHz,
        f_shift,
        x,
        left=x[0],
        right=x[-1]
    )


def park_single(x, ref, fHz):
    df_band = fHz[-1] - fHz[0]
    tau_max = PARK_MAX_SHIFT_FRAC * df_band

    best_err = np.inf
    best_tau = 0.0
    best_dS = 0.0

    taus = np.linspace(-tau_max, tau_max, PARK_NSTEPS)

    for tau in taus:
        x_shift = shift_interp(x, fHz, tau)

        dS = np.mean(ref - x_shift)

        y_try = x_shift + dS

        err = np.mean((ref - y_try) ** 2)

        if err < best_err:
            best_err = err
            best_tau = tau
            best_dS = dS

    y_comp = shift_interp(x, fHz, best_tau) + best_dS
    y_comp = moving_average(y_comp, PARK_SMOOTH_WIN)

    return y_comp


def compensar_park(df, fcols, fHz, y_ref_healthy):
    print("\n====================================================")
    print("APLICANDO PARK")
    print("====================================================")

    X_all = df[fcols].to_numpy(float)

    Y_comp = np.zeros_like(X_all)

    for i in range(len(X_all)):
        Y_comp[i] = park_single(
            x=X_all[i],
            ref=y_ref_healthy,
            fHz=fHz
        )

        if (i + 1) % 20 == 0 or (i + 1) == len(X_all):
            print(f"Park: {i+1}/{len(X_all)} curvas compensadas")

    df_comp = df.copy()
    df_comp[fcols] = Y_comp

    return df_comp


# ============================================================
# 8) MODELO TCNN
# ============================================================

class ResBlock1D(nn.Module):
    def __init__(self, cin, cout, stride=1):
        super().__init__()

        self.conv1 = nn.Conv1d(
            cin,
            cout,
            kernel_size=5,
            stride=stride,
            padding=2
        )

        self.norm1 = nn.GroupNorm(
            num_groups=min(8, cout),
            num_channels=cout
        )

        self.conv2 = nn.Conv1d(
            cout,
            cout,
            kernel_size=5,
            stride=1,
            padding=2
        )

        self.norm2 = nn.GroupNorm(
            num_groups=min(8, cout),
            num_channels=cout
        )

        if stride != 1 or cin != cout:
            self.skip = nn.Conv1d(
                cin,
                cout,
                kernel_size=1,
                stride=stride
            )
        else:
            self.skip = nn.Identity()

    def forward(self, x):
        identity = self.skip(x)

        y = self.conv1(x)
        y = self.norm1(y)
        y = F.gelu(y)

        y = self.conv2(y)
        y = self.norm2(y)

        y = y + identity
        y = F.gelu(y)

        return y


class ThermalCompensationNet(nn.Module):
    def __init__(
        self,
        n_points,
        n_classes,
        latent_dim=128,
        residual_scale=1.0
    ):
        super().__init__()

        self.n_points = int(n_points)
        self.n_classes = int(n_classes)
        self.latent_dim = int(latent_dim)
        self.residual_scale = float(residual_scale)

        self.down_factor = 16
        self.n_pad = int(
            np.ceil(self.n_points / self.down_factor) *
            self.down_factor
        )

        self.enc0 = ResBlock1D(1, 16, stride=1)
        self.enc1 = ResBlock1D(16, 32, stride=2)
        self.enc2 = ResBlock1D(32, 64, stride=2)
        self.enc3 = ResBlock1D(64, 128, stride=2)
        self.enc4 = ResBlock1D(128, 160, stride=2)

        self.bottleneck = ResBlock1D(160, 160, stride=1)

        self.dec3 = ResBlock1D(160 + 128, 128, stride=1)
        self.dec2 = ResBlock1D(128 + 64, 64, stride=1)
        self.dec1 = ResBlock1D(64 + 32, 32, stride=1)
        self.dec0 = ResBlock1D(32 + 16, 16, stride=1)

        self.out_delta = nn.Sequential(
            nn.Conv1d(16, 16, kernel_size=5, padding=2),
            nn.GELU(),
            nn.Conv1d(16, 1, kernel_size=5, padding=2)
        )

        self.pool = nn.AdaptiveAvgPool1d(1)

        self.latent_head = nn.Sequential(
            nn.Linear(160, 256),
            nn.GELU(),
            nn.Dropout(0.10),
            nn.Linear(256, latent_dim),
            nn.GELU()
        )

        self.damage_head = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.GELU(),
            nn.Dropout(0.10),
            nn.Linear(64, n_classes)
        )

        self.temp_head = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.GELU(),
            nn.Dropout(0.10),
            nn.Linear(64, 1)
        )

    def pad_input(self, x):
        L = x.shape[-1]

        if L == self.n_pad:
            return x

        pad_right = self.n_pad - L

        return F.pad(x, (0, pad_right), mode="replicate")

    def crop_output(self, x):
        return x[:, :, :self.n_points]

    def forward(self, x):
        x_in = x

        xp = self.pad_input(x)

        e0 = self.enc0(xp)
        e1 = self.enc1(e0)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)

        b = self.bottleneck(e4)

        pooled = self.pool(b).squeeze(-1)

        latent = self.latent_head(pooled)

        damage_logits = self.damage_head(latent)
        temp_pred = self.temp_head(latent)

        u3 = F.interpolate(
            b,
            size=e3.shape[-1],
            mode="linear",
            align_corners=False
        )
        u3 = torch.cat([u3, e3], dim=1)
        u3 = self.dec3(u3)

        u2 = F.interpolate(
            u3,
            size=e2.shape[-1],
            mode="linear",
            align_corners=False
        )
        u2 = torch.cat([u2, e2], dim=1)
        u2 = self.dec2(u2)

        u1 = F.interpolate(
            u2,
            size=e1.shape[-1],
            mode="linear",
            align_corners=False
        )
        u1 = torch.cat([u1, e1], dim=1)
        u1 = self.dec1(u1)

        u0 = F.interpolate(
            u1,
            size=e0.shape[-1],
            mode="linear",
            align_corners=False
        )
        u0 = torch.cat([u0, e0], dim=1)
        u0 = self.dec0(u0)

        raw_delta = self.out_delta(u0)
        raw_delta = self.crop_output(raw_delta)

        delta = self.residual_scale * torch.tanh(raw_delta)

        y_comp = x_in + ALPHA_COMP * delta

        return y_comp, delta, damage_logits, temp_pred


# ============================================================
# 9) PREPARAÇÃO PARA TREINO
# ============================================================

def preparar_dados_tcnn_ref_saudavel(
    df,
    fcols,
    healthy_by_temp,
    healthy_temps,
    y_ref_healthy
):
    X, Y, D_target, H_same_T, temp_healthy_used = (
        construir_targets_ref_saudavel_preservando_dano(
            df=df,
            fcols=fcols,
            healthy_by_temp=healthy_by_temp,
            healthy_temps=healthy_temps,
            y_ref_healthy=y_ref_healthy
        )
    )

    falhas_orig = df["falha"].to_numpy()

    classes = sorted(np.unique(falhas_orig))
    class_to_idx = {c: i for i, c in enumerate(classes)}
    idx_to_class = {i: c for c, i in class_to_idx.items()}

    y_class = np.array(
        [class_to_idx[c] for c in falhas_orig],
        dtype=int
    )

    T = df["temperatura_c"].to_numpy(float)

    T_aux = ((T - REF_TEMP) / 100.0).reshape(-1, 1)

    is_ref_healthy = (
        np.isclose(T, REF_TEMP) &
        (falhas_orig == 0)
    ).astype(float).reshape(-1, 1)

    return {
        "X": X,
        "Y": Y,
        "D_target": D_target,
        "H_same_T": H_same_T,
        "temp_healthy_used": temp_healthy_used,
        "y_class": y_class,
        "T_aux": T_aux,
        "is_ref_healthy": is_ref_healthy,
        "class_to_idx": class_to_idx,
        "idx_to_class": idx_to_class
    }


def make_train_val_split(y_class):
    indices = np.arange(len(y_class))

    values, counts = np.unique(y_class, return_counts=True)

    if len(values) > 1 and np.min(counts) >= 2:
        stratify = y_class
    else:
        stratify = None

    idx_train, idx_val = train_test_split(
        indices,
        test_size=0.20,
        random_state=42,
        stratify=stratify
    )

    return idx_train, idx_val


# ============================================================
# 10) TREINO TCNN
# ============================================================

def train_tcnn_ref_saudavel(
    df,
    fcols,
    healthy_by_temp,
    healthy_temps,
    y_ref_healthy
):
    print("\n====================================================")
    print("TREINANDO TCNN — REFERÊNCIA SEMPRE SAUDÁVEL")
    print("====================================================")
    print("Input da rede: curva medida")
    print("Temperatura NÃO entra como input")
    print("Falha NÃO entra como input")
    print("Referência base: saudável em REF_TEMP")
    print("Alvo: saudável REF + resíduo de dano preservado")
    print("Delta alvo: saudável REF - saudável na temperatura da amostra")

    dados = preparar_dados_tcnn_ref_saudavel(
        df=df,
        fcols=fcols,
        healthy_by_temp=healthy_by_temp,
        healthy_temps=healthy_temps,
        y_ref_healthy=y_ref_healthy
    )

    X = dados["X"]
    Y = dados["Y"]
    y_class = dados["y_class"]
    T_aux = dados["T_aux"]
    is_ref_healthy = dados["is_ref_healthy"]
    class_to_idx = dados["class_to_idx"]
    idx_to_class = dados["idx_to_class"]

    idx_train, idx_val = make_train_val_split(y_class)

    scaler = StandardScaler()

    scaler.fit(
        np.vstack([
            X[idx_train],
            Y[idx_train],
            dados["H_same_T"][idx_train],
            y_ref_healthy.reshape(1, -1)
        ])
    )

    Xs = scaler.transform(X)
    Ys = scaler.transform(Y)

    Ds = Ys - Xs

    y_ref_scaled = scaler.transform(
        y_ref_healthy.reshape(1, -1)
    )[0]

    residual_train = Ds[idx_train]

    residual_scale = float(
        np.percentile(
            np.abs(residual_train),
            RESIDUAL_PERCENTILE
        )
    )

    residual_scale = float(
        np.clip(
            residual_scale,
            RESIDUAL_SCALE_MIN,
            RESIDUAL_SCALE_MAX
        )
    )

    print(f"\nResidual scale usado no tanh: {residual_scale:.4f}")

    X_tensor = torch.tensor(Xs[:, None, :], dtype=torch.float32)
    Y_tensor = torch.tensor(Ys[:, None, :], dtype=torch.float32)
    D_tensor = torch.tensor(Ds[:, None, :], dtype=torch.float32)

    C_tensor = torch.tensor(y_class, dtype=torch.long)
    T_tensor = torch.tensor(T_aux, dtype=torch.float32)
    R_tensor = torch.tensor(is_ref_healthy, dtype=torch.float32)

    train_dataset = TensorDataset(
        X_tensor[idx_train],
        Y_tensor[idx_train],
        D_tensor[idx_train],
        C_tensor[idx_train],
        T_tensor[idx_train],
        R_tensor[idx_train]
    )

    val_dataset = TensorDataset(
        X_tensor[idx_val],
        Y_tensor[idx_val],
        D_tensor[idx_val],
        C_tensor[idx_val],
        T_tensor[idx_val],
        R_tensor[idx_val]
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS
    )

    device = "cuda" if torch.cuda.is_available() else "cpu"

    print(f"Dispositivo usado: {device}")

    model = ThermalCompensationNet(
        n_points=X.shape[1],
        n_classes=len(class_to_idx),
        latent_dim=LATENT_DIM,
        residual_scale=residual_scale
    ).to(device)

    ref_tensor = torch.tensor(
        y_ref_scaled[None, None, :],
        dtype=torch.float32,
        device=device
    )

    opt = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=1e-4
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt,
        mode="min",
        factor=0.5,
        patience=45
    )

    ce_loss = nn.CrossEntropyLoss()
    mse_loss = nn.MSELoss()
    huber_loss = nn.SmoothL1Loss()

    best_val = np.inf
    best_state = copy.deepcopy(model.state_dict())
    epochs_sem_melhora = 0

    history = {
        "epoch": [],
        "train_loss": [],
        "val_loss": [],
        "val_curve": [],
        "val_delta": [],
        "val_signature": [],
        "val_damage_acc": [],
        "lr": []
    }

    for ep in range(1, EPOCHS + 1):

        # ---------------- TREINO ----------------

        model.train()

        train_losses = []

        for xb, yb, db, cb, tb, rb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            db = db.to(device)
            cb = cb.to(device)
            tb = tb.to(device)
            rb = rb.to(device)

            opt.zero_grad()

            pred, delta, logits, tpred = model(xb)

            ref_batch = ref_tensor.expand_as(pred)

            pred_signature = pred - ref_batch
            true_signature = yb - ref_batch

            loss_curve = huber_loss(pred, yb)

            loss_delta_target = huber_loss(delta, db)
            loss_delta_deriv = derivative_loss(delta, db)

            loss_signature = huber_loss(
                pred_signature,
                true_signature
            )

            loss_signature_deriv = derivative_loss(
                pred_signature,
                true_signature
            )

            loss_corr = corr_loss_batch(pred, yb)

            loss_delta_energy = delta_energy_loss(delta)

            loss_dmg = ce_loss(logits, cb)

            if USE_TEMP_AUX_TARGET:
                loss_temp = mse_loss(tpred, tb)
            else:
                loss_temp = torch.tensor(0.0, device=device)

            ref_mask = rb.squeeze(1) > 0.5

            if torch.any(ref_mask):
                loss_identity = huber_loss(
                    pred[ref_mask],
                    xb[ref_mask]
                )
            else:
                loss_identity = torch.tensor(0.0, device=device)

            loss = (
                LAMBDA_CURVE * loss_curve
                + LAMBDA_DELTA_TARGET * loss_delta_target
                + LAMBDA_DELTA_DERIV * loss_delta_deriv
                + LAMBDA_SIGNATURE * loss_signature
                + LAMBDA_SIGNATURE_DERIV * loss_signature_deriv
                + LAMBDA_CORR * loss_corr
                + LAMBDA_DELTA_ENERGY * loss_delta_energy
                + LAMBDA_IDENTITY_REF * loss_identity
                + LAMBDA_DAMAGE * loss_dmg
                + LAMBDA_TEMP * loss_temp
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=5.0
            )

            opt.step()

            train_losses.append(loss.item())

        train_loss_mean = float(np.mean(train_losses))

        # ---------------- VALIDAÇÃO ----------------

        model.eval()

        val_losses = []
        val_curve_losses = []
        val_delta_losses = []
        val_signature_losses = []

        all_pred_class = []
        all_true_class = []

        with torch.no_grad():
            for xb, yb, db, cb, tb, rb in val_loader:
                xb = xb.to(device)
                yb = yb.to(device)
                db = db.to(device)
                cb = cb.to(device)
                tb = tb.to(device)
                rb = rb.to(device)

                pred, delta, logits, tpred = model(xb)

                ref_batch = ref_tensor.expand_as(pred)

                pred_signature = pred - ref_batch
                true_signature = yb - ref_batch

                loss_curve = huber_loss(pred, yb)

                loss_delta_target = huber_loss(delta, db)
                loss_delta_deriv = derivative_loss(delta, db)

                loss_signature = huber_loss(
                    pred_signature,
                    true_signature
                )

                loss_signature_deriv = derivative_loss(
                    pred_signature,
                    true_signature
                )

                loss_corr = corr_loss_batch(pred, yb)

                loss_delta_energy = delta_energy_loss(delta)

                loss_dmg = ce_loss(logits, cb)

                if USE_TEMP_AUX_TARGET:
                    loss_temp = mse_loss(tpred, tb)
                else:
                    loss_temp = torch.tensor(0.0, device=device)

                ref_mask = rb.squeeze(1) > 0.5

                if torch.any(ref_mask):
                    loss_identity = huber_loss(
                        pred[ref_mask],
                        xb[ref_mask]
                    )
                else:
                    loss_identity = torch.tensor(0.0, device=device)

                loss = (
                    LAMBDA_CURVE * loss_curve
                    + LAMBDA_DELTA_TARGET * loss_delta_target
                    + LAMBDA_DELTA_DERIV * loss_delta_deriv
                    + LAMBDA_SIGNATURE * loss_signature
                    + LAMBDA_SIGNATURE_DERIV * loss_signature_deriv
                    + LAMBDA_CORR * loss_corr
                    + LAMBDA_DELTA_ENERGY * loss_delta_energy
                    + LAMBDA_IDENTITY_REF * loss_identity
                    + LAMBDA_DAMAGE * loss_dmg
                    + LAMBDA_TEMP * loss_temp
                )

                val_losses.append(loss.item())
                val_curve_losses.append(loss_curve.item())
                val_delta_losses.append(loss_delta_target.item())
                val_signature_losses.append(loss_signature.item())

                pred_class = torch.argmax(logits, dim=1)

                all_pred_class.extend(
                    pred_class.cpu().numpy().tolist()
                )

                all_true_class.extend(
                    cb.cpu().numpy().tolist()
                )

        val_loss_mean = float(np.mean(val_losses))
        val_curve_mean = float(np.mean(val_curve_losses))
        val_delta_mean = float(np.mean(val_delta_losses))
        val_signature_mean = float(np.mean(val_signature_losses))

        val_acc = accuracy_score(
            all_true_class,
            all_pred_class
        )

        scheduler.step(val_loss_mean)

        current_lr = opt.param_groups[0]["lr"]

        history["epoch"].append(ep)
        history["train_loss"].append(train_loss_mean)
        history["val_loss"].append(val_loss_mean)
        history["val_curve"].append(val_curve_mean)
        history["val_delta"].append(val_delta_mean)
        history["val_signature"].append(val_signature_mean)
        history["val_damage_acc"].append(val_acc)
        history["lr"].append(current_lr)

        if val_loss_mean < best_val - 1e-7:
            best_val = val_loss_mean
            best_state = copy.deepcopy(model.state_dict())
            epochs_sem_melhora = 0
        else:
            epochs_sem_melhora += 1

        if ep == 1 or ep % 25 == 0:
            print(
                f"Epoch {ep:4d}/{EPOCHS} | "
                f"train={train_loss_mean:.6f} | "
                f"val={val_loss_mean:.6f} | "
                f"curve={val_curve_mean:.6f} | "
                f"delta={val_delta_mean:.6f} | "
                f"signature={val_signature_mean:.6f} | "
                f"dmg_acc={val_acc:.4f} | "
                f"lr={current_lr:.2e}"
            )

        if epochs_sem_melhora >= PATIENCE:
            print(
                f"Early stopping na epoch {ep}. "
                f"Melhor val_loss = {best_val:.6f}"
            )
            break

    model.load_state_dict(best_state)

    history = pd.DataFrame(history)

    extra = {
        "model": model,
        "scaler": scaler,
        "device": device,
        "history": history,
        "class_to_idx": class_to_idx,
        "idx_to_class": idx_to_class,
        "idx_train": idx_train,
        "idx_val": idx_val,
        "residual_scale": residual_scale,
        "y_ref_scaled": y_ref_scaled,
        "dados_preparo": dados
    }

    return extra


# ============================================================
# 11) APLICAR TCNN
# ============================================================

def aplicar_tcnn_em_todas_curvas(df, fcols, tcnn_extra):
    print("\n====================================================")
    print("APLICANDO TCNN EM TODAS AS CURVAS")
    print("====================================================")

    model = tcnn_extra["model"]
    scaler = tcnn_extra["scaler"]
    device = tcnn_extra["device"]
    idx_to_class = tcnn_extra["idx_to_class"]

    X = df[fcols].to_numpy(float)

    Xs = scaler.transform(X)

    X_tensor = torch.tensor(
        Xs[:, None, :],
        dtype=torch.float32
    )

    model.eval()

    preds_scaled = []
    delta_scaled_all = []
    logits_all = []
    temp_pred_all = []

    with torch.no_grad():
        for i in range(0, len(X_tensor), BATCH_SIZE):
            xb = X_tensor[i:i + BATCH_SIZE].to(device)

            pred, delta, logits, tpred = model(xb)

            preds_scaled.append(pred.cpu().numpy()[:, 0, :])
            delta_scaled_all.append(delta.cpu().numpy()[:, 0, :])
            logits_all.append(logits.cpu().numpy())
            temp_pred_all.append(tpred.cpu().numpy())

    preds_scaled = np.vstack(preds_scaled)
    delta_scaled_all = np.vstack(delta_scaled_all)
    logits_all = np.vstack(logits_all)
    temp_pred_all = np.vstack(temp_pred_all)

    Y_comp = scaler.inverse_transform(preds_scaled)

    pred_class_idx = np.argmax(logits_all, axis=1)

    pred_damage = np.array([
        idx_to_class[i] for i in pred_class_idx
    ])

    pred_temp_c = temp_pred_all[:, 0] * 100.0 + REF_TEMP

    df_comp = df.copy()
    df_comp[fcols] = Y_comp

    df_comp["falha_pred_aux"] = pred_damage
    df_comp["temperatura_pred_aux"] = pred_temp_c

    return df_comp


def compensar_tcnn_ref_saudavel(
    df,
    fcols,
    healthy_by_temp,
    healthy_temps,
    y_ref_healthy
):
    tcnn_extra = train_tcnn_ref_saudavel(
        df=df,
        fcols=fcols,
        healthy_by_temp=healthy_by_temp,
        healthy_temps=healthy_temps,
        y_ref_healthy=y_ref_healthy
    )

    df_comp = aplicar_tcnn_em_todas_curvas(
        df=df,
        fcols=fcols,
        tcnn_extra=tcnn_extra
    )

    return df_comp, tcnn_extra


# ============================================================
# 12) RESUMOS
# ============================================================

def resumo_geral(df_long):
    tabela = (
        df_long
        .groupby(["Metodo", "falha"])[[
            "RMSD",
            "CCDM",
            "DamageResidual_RMSD",
            "DamageResidual_CCDM",
            "Alteracao_RMSD"
        ]]
        .agg(["mean", "std", "min", "max"])
        .round(6)
    )

    return tabela


def resumo_por_temperatura(df_long):
    tabela = (
        df_long
        .groupby(["Metodo", "temperatura_c", "falha"])[[
            "RMSD",
            "CCDM",
            "DamageResidual_RMSD",
            "DamageResidual_CCDM",
            "Alteracao_RMSD"
        ]]
        .mean()
        .reset_index()
        .sort_values(["Metodo", "temperatura_c", "falha"])
    )

    return tabela


def checar_monotonicidade(df_long, metodos=("Original", "Park", "TCNN")):
    df_use = df_long[df_long["Metodo"].isin(metodos)].copy()

    registros = []

    for metodo in sorted(df_use["Metodo"].unique()):
        df_m = df_use[df_use["Metodo"] == metodo]

        temps = sorted(df_m["temperatura_c"].unique())

        for T in temps:
            df_t = df_m[np.isclose(df_m["temperatura_c"], T)]

            danos_presentes = set(df_t["falha"].unique())

            if not {0, 1, 2}.issubset(danos_presentes):
                continue

            for metrica in ["RMSD", "CCDM"]:
                medias = {}

                for d in [0, 1, 2]:
                    medias[d] = df_t.loc[
                        df_t["falha"] == d,
                        metrica
                    ].mean()

                ok = medias[0] < medias[1] < medias[2]

                registros.append({
                    "Metodo": metodo,
                    "Temperatura": T,
                    "Metrica": metrica,
                    "D0": medias[0],
                    "D1": medias[1],
                    "D2": medias[2],
                    "Monotonico_D0_D1_D2": ok
                })

    df_mono = pd.DataFrame(registros)

    if len(df_mono) == 0:
        return df_mono, pd.DataFrame()

    resumo = (
        df_mono
        .groupby(["Metodo", "Metrica"])["Monotonico_D0_D1_D2"]
        .mean()
        .mul(100)
        .reset_index()
        .rename(columns={
            "Monotonico_D0_D1_D2": "Percentual_monotonico_%"
        })
    )

    return df_mono, resumo


# ============================================================
# 13) GRÁFICOS
# ============================================================

def selecionar_indice_por_dano_temperatura(
    df,
    falha,
    temperatura,
    ocorrencia=0
):
    df_d = df[df["falha"] == falha].copy()

    if len(df_d) == 0:
        raise ValueError(f"Nenhuma curva encontrada para falha = {falha}")

    temps_disponiveis = np.array(
        sorted(df_d["temperatura_c"].unique()),
        dtype=float
    )

    temp_usada = temps_disponiveis[
        np.argmin(np.abs(temps_disponiveis - temperatura))
    ]

    df_dt = df_d[np.isclose(df_d["temperatura_c"], temp_usada)].copy()

    if len(df_dt) == 0:
        raise ValueError(
            f"Nenhuma curva encontrada para falha={falha} "
            f"na temperatura {temp_usada} °C."
        )

    if ocorrencia >= len(df_dt):
        print(
            f"AVISO: falha={falha}, T={temp_usada} °C possui apenas "
            f"{len(df_dt)} curva(s). Usando ocorrência 0."
        )
        ocorrencia = 0

    idx = df_dt.index[ocorrencia]

    return idx, temp_usada


def plot_curvas_sem_park_tres_danos(
    df_base,
    df_tcnn,
    y_ref_healthy,
    healthy_by_temp,
    healthy_temps,
    fcols,
    fhz,
    temperatura_escolhida=55,
    danos=(0, 1, 2),
    ocorrencia=0,
    salvar=True,
    show=True
):
    """
    Curvas SEM Park.

    Em cada dano:
    - referência saudável em REF_TEMP
    - saudável na temperatura original
    - curva original
    - curva TCNN

    A referência visual é sempre saudável.
    """

    aplicar_estilo_artigo()

    fhz_khz = fhz / 1e3

    fig, axes = plt.subplots(
        len(danos),
        1,
        figsize=(13, 5.2 * len(danos)),
        dpi=300,
        sharex=True
    )

    if len(danos) == 1:
        axes = [axes]

    print("\n====================================================")
    print("CURVAS SEM PARK")
    print("Referência sempre saudável")
    print("====================================================")

    for ax, dano in zip(axes, danos):
        idx_show, temp_usada = selecionar_indice_por_dano_temperatura(
            df=df_base,
            falha=dano,
            temperatura=temperatura_escolhida,
            ocorrencia=ocorrencia
        )

        y_original = df_base.loc[idx_show, fcols].to_numpy(float)
        y_tcnn = df_tcnn.loc[idx_show, fcols].to_numpy(float)

        h_T, T_h_used = get_nearest_healthy_curve(
            healthy_by_temp=healthy_by_temp,
            healthy_temps=healthy_temps,
            temperatura=temp_usada
        )

        assinatura_original = y_original - h_T
        assinatura_tcnn = y_tcnn - y_ref_healthy

        print("\n----------------------------------------------------")
        print(f"Dano {dano}")
        print(f"Índice usado: {idx_show}")
        print(f"Temperatura da curva: {temp_usada} °C")
        print(f"Temperatura saudável usada para resíduo: {T_h_used} °C")
        print("Original contra saudável REF:")
        print(f"  RMSD = {rmsd(y_original, y_ref_healthy):.6f}")
        print(f"  CCDM = {ccdm(y_original, y_ref_healthy):.6f}")
        print("TCNN contra saudável REF:")
        print(f"  RMSD = {rmsd(y_tcnn, y_ref_healthy):.6f}")
        print(f"  CCDM = {ccdm(y_tcnn, y_ref_healthy):.6f}")
        print("Preservação da assinatura:")
        print(f"  RMSD residual = {rmsd(assinatura_tcnn, assinatura_original):.6f}")
        print(f"  CCDM residual = {ccdm(assinatura_tcnn, assinatura_original):.6f}")

        ax.plot(
            fhz_khz,
            y_ref_healthy,
            "--",
            color="black",
            linewidth=1.5,
            label=f"Referência saudável {REF_TEMP} °C"
        )

        ax.plot(
            fhz_khz,
            h_T,
            "-.",
            color="gray",
            linewidth=1.2,
            label=f"Saudável {formatar_temp(T_h_used)} °C"
        )

        ax.plot(
            fhz_khz,
            y_original,
            color="tab:red",
            linewidth=1.1,
            alpha=0.75,
            label=f"Original — Dano {dano} — {formatar_temp(temp_usada)} °C"
        )

        ax.plot(
            fhz_khz,
            y_tcnn,
            color="tab:blue",
            linewidth=2.0,
            label="TCNN compensada"
        )

        ax.set_ylabel("Parte real da impedância")

        ax.set_title(
            f"Dano {dano} — {formatar_temp(temp_usada)} °C "
            f"→ compensado para referência saudável {REF_TEMP} °C"
        )

        ax.grid(False)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        ax.legend(
            frameon=True,
            facecolor="white",
            edgecolor="none"
        )

    axes[-1].set_xlabel("Frequência (kHz)")

    plt.tight_layout()

    if salvar:
        nome = (
            f"Curvas_sem_Park_ref_saudavel_"
            f"Temp_{formatar_temp(temperatura_escolhida)}C"
        )

        salvar_figura(fig, nome)

    if show:
        plt.show()
    else:
        plt.close(fig)


def escolher_temperaturas_validas(
    df_long,
    metodos=("Original", "Park", "TCNN"),
    n_temps=8,
    seed=42,
    temperaturas_especificas=None
):
    valid_temps = None

    for metodo in metodos:
        for dano in [0, 1, 2]:
            temps = set(
                df_long.loc[
                    (df_long["Metodo"] == metodo) &
                    (df_long["falha"] == dano),
                    "temperatura_c"
                ].unique()
            )

            if valid_temps is None:
                valid_temps = temps
            else:
                valid_temps = valid_temps & temps

    valid_temps = sorted(list(valid_temps))

    if len(valid_temps) == 0:
        raise ValueError(
            "Nenhuma temperatura contém os três danos "
            "em todos os métodos escolhidos."
        )

    if temperaturas_especificas is not None:
        out = []

        for T in temperaturas_especificas:
            if any(np.isclose(T, Tv) for Tv in valid_temps):
                arr = np.asarray(valid_temps, dtype=float)
                out.append(float(arr[np.argmin(np.abs(arr - T))]))
            else:
                print(f"AVISO: temperatura {T} °C ignorada.")

        valid_temps = sorted(list(set(out)))

        if len(valid_temps) == 0:
            raise ValueError("Nenhuma temperatura escolhida é válida.")

        return valid_temps

    if len(valid_temps) > n_temps:
        rng = np.random.default_rng(seed)
        valid_temps = sorted(
            rng.choice(valid_temps, n_temps, replace=False)
        )

    return valid_temps


def valores_medios_por_temp_dano_metodo(
    df_long,
    metodo,
    dano,
    temperaturas,
    metrica
):
    vals = []

    for T in temperaturas:
        mask = (
            (df_long["Metodo"] == metodo) &
            (df_long["falha"] == dano) &
            (np.isclose(df_long["temperatura_c"], T))
        )

        if np.any(mask):
            vals.append(df_long.loc[mask, metrica].mean())
        else:
            vals.append(np.nan)

    return vals


def plot_barras_original_park_tcnn(
    df_long,
    metricas=("RMSD", "CCDM"),
    temperaturas=None,
    n_temps=8,
    seed=42,
    nome_base="Barras_Original_Park_TCNN_RMSD_CCDM",
    salvar=True,
    show=True
):
    aplicar_estilo_artigo()

    metodos = ["Original", "Park", "TCNN"]
    danos = [0, 1, 2]

    df_use = df_long[df_long["Metodo"].isin(metodos)].copy()

    temps_validas = escolher_temperaturas_validas(
        df_long=df_use,
        metodos=metodos,
        n_temps=n_temps,
        seed=seed,
        temperaturas_especificas=temperaturas
    )

    x = np.arange(len(temps_validas))

    bar_w = 0.08
    gap = 0.08

    original_offsets = np.array([0, 1, 2]) * bar_w
    park_offsets = (3 * bar_w + gap) + np.array([0, 1, 2]) * bar_w
    tcnn_offsets = (6 * bar_w + 2 * gap) + np.array([0, 1, 2]) * bar_w

    all_offsets = np.concatenate([
        original_offsets,
        park_offsets,
        tcnn_offsets
    ])

    x_center = x + np.mean(all_offsets)

    colors = {
        0: "tab:blue",
        1: "tab:orange",
        2: "tab:red"
    }

    fig, axes = plt.subplots(
        1,
        len(metricas),
        figsize=(11 * len(metricas), 7.2),
        dpi=300
    )

    if len(metricas) == 1:
        axes = [axes]

    for ax in axes:
        ax.grid(False)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    for j, metrica in enumerate(metricas):
        ax = axes[j]

        for i, dano in enumerate(danos):
            vals_original = valores_medios_por_temp_dano_metodo(
                df_long=df_use,
                metodo="Original",
                dano=dano,
                temperaturas=temps_validas,
                metrica=metrica
            )

            vals_park = valores_medios_por_temp_dano_metodo(
                df_long=df_use,
                metodo="Park",
                dano=dano,
                temperaturas=temps_validas,
                metrica=metrica
            )

            vals_tcnn = valores_medios_por_temp_dano_metodo(
                df_long=df_use,
                metodo="TCNN",
                dano=dano,
                temperaturas=temps_validas,
                metrica=metrica
            )

            ax.bar(
                x + original_offsets[i],
                vals_original,
                width=bar_w,
                color=colors[dano],
                alpha=0.35,
                edgecolor="black",
                linewidth=0.7,
                label=f"Original — Dano {dano}" if j == 0 else None
            )

            ax.bar(
                x + park_offsets[i],
                vals_park,
                width=bar_w,
                color=colors[dano],
                alpha=0.65,
                edgecolor="black",
                linewidth=0.7,
                label=f"Park — Dano {dano}" if j == 0 else None
            )

            ax.bar(
                x + tcnn_offsets[i],
                vals_tcnn,
                width=bar_w,
                color=colors[dano],
                alpha=1.00,
                edgecolor="black",
                linewidth=0.7,
                label=f"TCNN — Dano {dano}" if j == 0 else None
            )

        ax.set_ylabel(metrica)
        ax.set_xlabel("Temperatura (°C)", labelpad=10)

        ax.set_xticks(x_center)
        ax.set_xticklabels([
            formatar_temp(T) for T in temps_validas
        ])

        letra = chr(ord("a") + j)

        ax.text(
            0.5,
            -0.30,
            f"({letra}) {metrica}",
            transform=ax.transAxes,
            ha="center",
            va="top",
            fontsize=22
        )

    handles, labels = axes[0].get_legend_handles_labels()

    fig.legend(
        handles,
        labels,
        loc="upper center",
        ncol=3,
        frameon=False,
        bbox_to_anchor=(0.5, 1.11)
    )

    plt.tight_layout(rect=[0, 0.10, 1, 0.90])

    if salvar:
        salvar_figura(fig, nome_base)

    if show:
        plt.show()
    else:
        plt.close(fig)


def plot_histogramas_distribuicao(
    df_long,
    metricas=("RMSD", "CCDM"),
    bins=18,
    nome_base="Histogramas_Distribuicao_Original_Park_TCNN",
    salvar=True,
    show=True
):
    aplicar_estilo_artigo()

    metodos = ["Original", "Park", "TCNN"]
    danos = [0, 1, 2]

    df_use = df_long[df_long["Metodo"].isin(metodos)].copy()

    cores = {
        "Original": "tab:gray",
        "Park": "tab:red",
        "TCNN": "tab:blue"
    }

    alphas = {
        "Original": 0.35,
        "Park": 0.50,
        "TCNN": 0.65
    }

    fig, axes = plt.subplots(
        len(danos),
        len(metricas),
        figsize=(8.5 * len(metricas), 5.2 * len(danos)),
        dpi=300
    )

    if len(metricas) == 1:
        axes = np.array([[ax] for ax in axes])

    for i, dano in enumerate(danos):
        for j, metrica in enumerate(metricas):
            ax = axes[i, j]

            for metodo in metodos:
                vals = df_use.loc[
                    (df_use["Metodo"] == metodo) &
                    (df_use["falha"] == dano),
                    metrica
                ].dropna().to_numpy(float)

                if len(vals) == 0:
                    continue

                ax.hist(
                    vals,
                    bins=bins,
                    alpha=alphas[metodo],
                    color=cores[metodo],
                    edgecolor="black",
                    linewidth=0.7,
                    label=metodo
                )

                ax.axvline(
                    np.mean(vals),
                    color=cores[metodo],
                    linestyle="--",
                    linewidth=2
                )

            ax.set_xlabel(metrica)
            ax.set_ylabel("Frequência")
            ax.set_title(f"{metrica} — Dano {dano}")

            ax.grid(False)
            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)

            if i == 0 and j == 0:
                ax.legend(
                    frameon=True,
                    facecolor="white",
                    edgecolor="none"
                )

    plt.tight_layout()

    if salvar:
        salvar_figura(fig, nome_base)

    if show:
        plt.show()
    else:
        plt.close(fig)


def plot_tcnn_loss(history, salvar=True, show=True):
    aplicar_estilo_artigo()

    fig, ax = plt.subplots(figsize=(10, 5), dpi=300)

    ax.plot(
        history["epoch"],
        history["train_loss"],
        linewidth=2,
        label="Treino"
    )

    ax.plot(
        history["epoch"],
        history["val_loss"],
        linewidth=2,
        label="Validação total"
    )

    ax.plot(
        history["epoch"],
        history["val_curve"],
        linewidth=2,
        label="Validação curva"
    )

    ax.plot(
        history["epoch"],
        history["val_delta"],
        linewidth=2,
        label="Validação delta térmico"
    )

    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_title("Treinamento da TCNN — referência saudável")

    ax.grid(False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.legend(
        frameon=True,
        facecolor="white",
        edgecolor="none"
    )

    plt.tight_layout()

    if salvar:
        salvar_figura(fig, "TCNN_loss_ref_saudavel")

    if show:
        plt.show()
    else:
        plt.close(fig)


def avaliar_auxiliares(df_comp):
    y_true = df_comp["falha"].to_numpy()
    y_pred = df_comp["falha_pred_aux"].to_numpy()

    print("\n==================== CLASSIFICAÇÃO AUXILIAR DE DANO ====================")
    print("A falha NÃO foi input da rede.")
    print("Matriz de confusão:")
    print(confusion_matrix(y_true, y_pred))
    print(f"ACC = {accuracy_score(y_true, y_pred):.4f}")
    print(f"F1 macro = {f1_score(y_true, y_pred, average='macro'):.4f}")

    if "temperatura_pred_aux" in df_comp.columns:
        temp_true = df_comp["temperatura_c"].to_numpy(float)
        temp_pred = df_comp["temperatura_pred_aux"].to_numpy(float)

        mae = np.mean(np.abs(temp_true - temp_pred))
        rmse = np.sqrt(np.mean((temp_true - temp_pred) ** 2))

        print("\n==================== PREDIÇÃO AUXILIAR DE TEMPERATURA ====================")
        print("A temperatura NÃO foi input da rede.")
        print(f"MAE temperatura = {mae:.4f} °C")
        print(f"RMSE temperatura = {rmse:.4f} °C")


# ============================================================
# 14) EXECUÇÃO PRINCIPAL
# ============================================================

def executar_tcnn_ref_saudavel():
    timings = {}

    print("====================================================")
    print("TCNN COM REFERÊNCIA SEMPRE SAUDÁVEL")
    print("====================================================")

    t0 = time.time()

    df = pd.read_pickle(ARQ_BASE).reset_index(drop=True)

    required_cols = {"temperatura_c", "falha"}

    missing = required_cols - set(df.columns)

    if len(missing) > 0:
        raise ValueError(f"Colunas obrigatórias ausentes: {missing}")

    fcols, fhz = get_freq_columns(
        df,
        FREQ_MIN_KHZ,
        FREQ_MAX_KHZ
    )

    if len(fcols) == 0:
        raise ValueError(
            "Nenhuma coluna de frequência encontrada na faixa escolhida."
        )

    healthy_by_temp, healthy_temps, y_ref_healthy, ref_temp_used = (
        get_healthy_references_by_temperature(
            df=df,
            fcols=fcols,
            ref_temp=REF_TEMP
        )
    )

    timings["load_reference"] = time.time() - t0

    print(f"\nTotal de amostras: {len(df)}")
    print(f"Classes de dano: {sorted(df['falha'].unique())}")
    print(f"Amostras saudáveis: {len(df[df['falha'] == 0])}")
    print(f"Faixa usada: {FREQ_MIN_KHZ}-{FREQ_MAX_KHZ} kHz")
    print(f"Número de pontos de frequência: {len(fcols)}")
    print(f"Temperatura de referência desejada: {REF_TEMP} °C")
    print(f"Temperatura saudável usada como REF: {ref_temp_used} °C")

    # ---------------- ORIGINAL ----------------

    t0 = time.time()

    df_original_metricas = calcular_metricas_com_preservacao(
        df_curvas=df,
        df_original=df,
        fcols=fcols,
        y_ref_healthy=y_ref_healthy,
        healthy_by_temp=healthy_by_temp,
        healthy_temps=healthy_temps,
        metodo="Original"
    )

    timings["original_metrics"] = time.time() - t0

    # ---------------- PARK ----------------

    t0 = time.time()

    df_park_curvas = compensar_park(
        df=df,
        fcols=fcols,
        fHz=fhz,
        y_ref_healthy=y_ref_healthy
    )

    df_park_metricas = calcular_metricas_com_preservacao(
        df_curvas=df_park_curvas,
        df_original=df,
        fcols=fcols,
        y_ref_healthy=y_ref_healthy,
        healthy_by_temp=healthy_by_temp,
        healthy_temps=healthy_temps,
        metodo="Park"
    )

    timings["park"] = time.time() - t0

    # ---------------- TCNN ----------------

    t0 = time.time()

    df_tcnn_curvas, tcnn_extra = compensar_tcnn_ref_saudavel(
        df=df,
        fcols=fcols,
        healthy_by_temp=healthy_by_temp,
        healthy_temps=healthy_temps,
        y_ref_healthy=y_ref_healthy
    )

    df_tcnn_metricas = calcular_metricas_com_preservacao(
        df_curvas=df_tcnn_curvas,
        df_original=df,
        fcols=fcols,
        y_ref_healthy=y_ref_healthy,
        healthy_by_temp=healthy_by_temp,
        healthy_temps=healthy_temps,
        metodo="TCNN"
    )

    timings["tcnn"] = time.time() - t0

    # ---------------- JUNTAR ----------------

    df_long = pd.concat(
        [
            df_original_metricas,
            df_park_metricas,
            df_tcnn_metricas
        ],
        axis=0,
        ignore_index=True
    )

    tabela_resumo = resumo_geral(df_long)
    tabela_temp = resumo_por_temperatura(df_long)

    df_mono, resumo_mono = checar_monotonicidade(
        df_long,
        metodos=("Original", "Park", "TCNN")
    )

    # ---------------- SALVAR ----------------

    df_long.to_csv(
        os.path.join(OUTPUT_DIR, "df_long_metricas.csv"),
        index=False
    )

    tabela_temp.to_csv(
        os.path.join(OUTPUT_DIR, "resumo_por_temperatura.csv"),
        index=False
    )

    df_mono.to_csv(
        os.path.join(OUTPUT_DIR, "monotonicidade.csv"),
        index=False
    )

    resumo_mono.to_csv(
        os.path.join(OUTPUT_DIR, "resumo_monotonicidade.csv"),
        index=False
    )

    tcnn_extra["history"].to_csv(
        os.path.join(OUTPUT_DIR, "historico_loss_tcnn.csv"),
        index=False
    )

    df_tcnn_curvas[[
        "temperatura_c",
        "falha",
        "falha_pred_aux",
        "temperatura_pred_aux"
    ]].to_csv(
        os.path.join(OUTPUT_DIR, "predicoes_auxiliares.csv"),
        index=False
    )

    print("\n==================== RESUMO GERAL ====================")
    print(tabela_resumo)

    print("\n==================== MONOTONICIDADE ====================")
    print(resumo_mono)

    print("\n==================== TEMPOS ====================")
    for k, v in timings.items():
        print(f"{k:28s}: {v:.3f} s")

    avaliar_auxiliares(df_tcnn_curvas)

    print("\n✅ Execução concluída.")

    return {
        "df_base": df,
        "df_park_curvas": df_park_curvas,
        "df_tcnn_curvas": df_tcnn_curvas,
        "df_original_metricas": df_original_metricas,
        "df_park_metricas": df_park_metricas,
        "df_tcnn_metricas": df_tcnn_metricas,
        "df_long": df_long,
        "tabela_resumo": tabela_resumo,
        "tabela_temp": tabela_temp,
        "df_mono": df_mono,
        "resumo_mono": resumo_mono,
        "healthy_by_temp": healthy_by_temp,
        "healthy_temps": healthy_temps,
        "y_ref_healthy": y_ref_healthy,
        "ref_temp_used": ref_temp_used,
        "fcols": fcols,
        "fhz": fhz,
        "tcnn_extra": tcnn_extra,
        "tcnn_history": tcnn_extra["history"],
        "timings": timings
    }


# ============================================================
# 15) RODAR TUDO
# ============================================================

resultados = executar_tcnn_ref_saudavel()

df_base = resultados["df_base"]
df_park_curvas = resultados["df_park_curvas"]
df_tcnn_curvas = resultados["df_tcnn_curvas"]

df_original_metricas = resultados["df_original_metricas"]
df_park_metricas = resultados["df_park_metricas"]
df_tcnn_metricas = resultados["df_tcnn_metricas"]

df_long = resultados["df_long"]

tabela_resumo = resultados["tabela_resumo"]
tabela_temp = resultados["tabela_temp"]
df_mono = resultados["df_mono"]
resumo_mono = resultados["resumo_mono"]

healthy_by_temp = resultados["healthy_by_temp"]
healthy_temps = resultados["healthy_temps"]
y_ref_healthy = resultados["y_ref_healthy"]
ref_temp_used = resultados["ref_temp_used"]

fcols = resultados["fcols"]
fhz = resultados["fhz"]

tcnn_extra = resultados["tcnn_extra"]
tcnn_history = resultados["tcnn_history"]
timings = resultados["timings"]


# ============================================================
# 16) GERAR FIGURAS
# ============================================================

# Curvas sem Park:
# Original + saudável na temperatura da amostra + saudável REF + TCNN.
plot_curvas_sem_park_tres_danos(
    df_base=df_base,
    df_tcnn=df_tcnn_curvas,
    y_ref_healthy=y_ref_healthy,
    healthy_by_temp=healthy_by_temp,
    healthy_temps=healthy_temps,
    fcols=fcols,
    fhz=fhz,
    temperatura_escolhida=TEMP_ESCOLHIDA,
    danos=DANOS_PLOTAR,
    ocorrencia=OCORRENCIA_CURVA,
    salvar=True,
    show=True
)

# Barras principais:
# RMSD e CCDM contra referência saudável.
plot_barras_original_park_tcnn(
    df_long=df_long,
    metricas=("RMSD", "CCDM"),
    temperaturas=TEMPERATURAS_BARRAS,
    n_temps=N_TEMPS_BARRAS,
    seed=42,
    nome_base="Barras_Original_Park_TCNN_RMSD_CCDM_ref_saudavel",
    salvar=True,
    show=True
)

# Barras de preservação:
# Quanto menor, melhor preservou a assinatura de dano.
plot_barras_original_park_tcnn(
    df_long=df_long,
    metricas=("DamageResidual_RMSD", "DamageResidual_CCDM"),
    temperaturas=TEMPERATURAS_BARRAS,
    n_temps=N_TEMPS_BARRAS,
    seed=42,
    nome_base="Barras_Preservacao_Assinatura_Dano",
    salvar=True,
    show=True
)

# Histogramas principais.
plot_histogramas_distribuicao(
    df_long=df_long,
    metricas=("RMSD", "CCDM"),
    bins=HIST_BINS,
    nome_base="Histogramas_Distribuicao_RMSD_CCDM_ref_saudavel",
    salvar=True,
    show=True
)

# Histogramas de preservação.
plot_histogramas_distribuicao(
    df_long=df_long,
    metricas=("DamageResidual_RMSD", "DamageResidual_CCDM"),
    bins=HIST_BINS,
    nome_base="Histogramas_Distribuicao_Preservacao_Assinatura_Dano",
    salvar=True,
    show=True
)

# Loss da TCNN.
plot_tcnn_loss(
    history=tcnn_history,
    salvar=True,
    show=True
)


# ============================================================
# 17) MOSTRAR TABELAS
# ============================================================

print("\n==================== TABELA RESUMO ====================")
display(tabela_resumo)

print("\n==================== RESUMO POR TEMPERATURA ====================")
display(tabela_temp)

print("\n==================== MONOTONICIDADE POR TEMPERATURA ====================")
display(df_mono)

print("\n==================== RESUMO DA MONOTONICIDADE ====================")
display(resumo_mono)

print("\n✅ Todos os gráficos foram gerados e salvos.")
print(f"Pasta de saída: {OUTPUT_DIR}")
